In [0]:
# ============================================================
# GOLD LAYER - DIM_ACCOUNT - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# ── Load ──
df_account = spark.read.format("delta").load(f"{SILVER}/account_master")

# ── Transform ──
dim_account = df_account.select(
    F.col("account_id"),
    F.col("customer_id"),
    F.col("product_type"),
    F.col("account_open_date"),
    F.col("credit_limit"),
    F.col("interest_rate"),
    F.col("account_status"),
    F.when(F.col("credit_limit") < 200000, F.lit("Basic"))
     .when(F.col("credit_limit") < 500000, F.lit("Standard"))
     .when(F.col("credit_limit") < 800000, F.lit("Premium"))
     .otherwise(F.lit("Elite"))
     .alias("credit_tier"),
    F.when(F.col("interest_rate") < 12, F.lit("Low Rate"))
     .when(F.col("interest_rate") < 18, F.lit("Mid Rate"))
     .otherwise(F.lit("High Rate"))
     .alias("rate_band"),
    F.current_timestamp().alias("gold_created_at")
)

# ── Write ──
dim_account.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/dim_account")

print(f"\n{'='*55}")
print(f"✅ dim_account : {dim_account.count():,} rows")
print(f"📁 Written to: {GOLD}/dim_account")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")

✅ dim_account : 150,000 rows
